In [336]:
import numpy as np
import pandas as pd

# П.1 - п.8

## Обработка датасетов

In [316]:
# Обработка датасетов ru_regions.csv и Датасет_ГАС_Управление.csv

dct = {
    "удмуртская республика": "Республика Удмуртия",
    "республика северная осетия-алания": "Республика Северная Осетия - Алания",
    "чеченская республика": "Республика Чечня",
    "чувашская республика - чувашия": "Республика Чувашия",
    "кабардино-балкарская республика": "Республика Кабардино-Балкария",
    "карачаево-черкесская республика": "Республика Карачаево-Черкесия",
    "ханты-мансийский автономный округ - югра": "Ханты-Мансийский (Югра) автономный округ",
    "кемеровская область - кузбасс": "Кемеровская область",
}

# Обработка df1
df1 = pd.read_csv("ru_regions.csv", sep=";")
for row in df1.itertuples(index=True):
    idx = row.Index
    if row.region[0].islower():
        df1.loc[idx, "region"] = row.region[0].upper() + row.region[1:]

# Обработка df2
df2 = pd.read_csv("Датасет_ГАС_Управление.csv", sep=";", encoding="utf-8-sig")
df2 = df2.drop(columns=["Бюджет"])
df2["Территория"] = df2["Территория"].str.replace(r"г\.\s*", "", regex=True)
df2["Территория_lower"] = df2["Территория"].str.lower().str.strip()
df2["Территория_lower"] = df2["Территория_lower"].replace(dct)
for row in df2.itertuples(index=True):
    idx = row.Index
    if not row.Территория_lower[0].islower():
        df2.loc[idx, "Территория"] = row.Территория_lower

arr1 = df1["region"].str.lower().unique()
arr2 = df2["Территория"].str.lower().unique()

common = np.intersect1d(arr1, arr2)
print("Совпадений:", common.shape)

df2_filtered = df2[df2["Территория"].str.lower().isin(common)].copy()
print("Объектов:", df2_filtered["Территория"].unique().shape)

grouped = (df2_filtered.groupby("Территория", as_index=False)["Объем поступлений по доходам"].sum())
grouped

out_name_1 = "ru_regions_edited.csv"
df1.to_csv(out_name_1, sep=";", index=False, encoding="utf-8-sig")
print(f"Файл сохранён: {out_name_1}")

out_name_2 = "ru_regions_income.csv"
grouped.to_csv(out_name_2, sep=";", index=False, encoding="utf-8-sig")
print(f"Файл сохранён: {out_name_2}")

Совпадений: (85,)
Объектов: (85,)
Файл сохранён: ru_regions_edited.csv
Файл сохранён: ru_regions_income.csv


## Проверка

In [317]:
df1 = pd.read_csv(r"ru_regions_edited.csv", sep=";", encoding="utf-8-sig")
df1.head(3)

,name,type,id,region,coords_type,coords
0,Ростовская,Область,64,Ростовская область,ru_regions_poly,"[[[50.21447689972731,41.3502545996028],[50.199..."
1,Рязанская,Область,65,Рязанская область,ru_regions_poly,"[[[55.36424399994077,40.210444700223796],[55.3..."
2,Сахалинская,Область,69,Сахалинская область,ru_regions_poly,"[[[46.4051481123399,142.82388437012776],[46.39..."


In [318]:
df3 = pd.read_csv(r"ru_regions_income.csv", sep=";", encoding="utf-8-sig")
df3.head(3)

,Территория,Объем поступлений по доходам
0,Алтайский край,537363.63
1,Амурская область,337575.48
2,Архангельская область,685979.04


In [319]:
arr_1 = df1["region"].unique()
arr_3 = df3["Территория"].unique()
for el in arr_3:
    if el not in arr_1:
        print(el, "-->", el in arr_1)
for el in arr_1:
    if el not in arr_3:
        print(el, "-->", el in arr_3)

# П.9 - п.12

## Загрузка датасета и обзор датасета

In [348]:
data = pd.read_csv("Доходы_регионов_сокр.csv", sep=",", index_col=0)
data.head(3)

,year,month,income_level,income_part,plan,execution_regional,growth_rate_regional,object_name,okato,oktmo,object_level
0,2015,5,0,ИТОГО ДОХОДОВ,65790667.0,2.468728e+07,1.108331,Белгородская область,14000000,14000000,регион
1,2015,5,1,НАЛОГОВЫЕ И НЕНАЛОГОВЫЕ ДОХОДЫ,58709651.0,1.697837e+07,1.096444,Белгородская область,14000000,14000000,регион
2,2015,5,2,"НАЛОГИ НА ПРИБЫЛЬ, ДОХОДЫ",34589166.0,1.022939e+07,1.102889,Белгородская область,14000000,14000000,регион


In [349]:
data.describe()

,year,month,income_level,plan,execution_regional,growth_rate_regional,okato,oktmo
count,908163.000000,908163.000000,908163.000000,9.081630e+05,6.849330e+05,8.174790e+05,9.081630e+05,9.081630e+05
mean,2020.192382,6.533893,2.692415,6.586317e+06,5.352065e+06,-1.591347e+13,5.184749e+07,5.187174e+07
std,3.057011,3.394988,0.575207,4.880201e+07,4.973275e+07,6.155772e+15,2.882463e+07,2.882661e+07
min,2015.000000,1.000000,0.000000,-3.860757e+06,-2.210167e+07,-2.726128e+18,1.000000e+06,1.000000e+06
25%,2018.000000,4.000000,2.000000,2.500000e+02,5.605140e+02,7.742766e-01,2.700000e+07,2.700000e+07
50%,2020.000000,7.000000,3.000000,3.663733e+04,3.076619e+04,1.054375e+00,5.300000e+07,5.300000e+07
75%,2023.000000,9.000000,3.000000,8.265790e+05,8.378772e+05,1.379483e+00,7.700000e+07,7.700000e+07
max,2025.000000,12.000000,3.000000,5.125073e+09,5.170180e+09,8.551204e+16,9.900000e+07,9.900000e+07


In [350]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 908163 entries, 0 to 5860902
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   year                  908163 non-null  int64  
 1   month                 908163 non-null  int64  
 2   income_level          908163 non-null  int64  
 3   income_part           908163 non-null  object 
 4   plan                  908163 non-null  float64
 5   execution_regional    684933 non-null  float64
 6   growth_rate_regional  817479 non-null  float64
 7   object_name           908163 non-null  object 
 8   okato                 908163 non-null  int64  
 9   oktmo                 908163 non-null  int64  
 10  object_level          908163 non-null  object 
dtypes: float64(3), int64(5), object(3)
memory usage: 83.1+ MB


### Категориальные признаки

In [351]:
str_cols = data.select_dtypes(include="object").columns
str_cols

Index(['income_part', 'object_name', 'object_level'], dtype='object')

In [352]:
for col in str_cols:
    print(f"Количество уникальных значений в стоблце {col}: {data[col].nunique()}")

Количество уникальных значений в стоблце income_part: 187
Количество уникальных значений в стоблце object_name: 85
Количество уникальных значений в стоблце object_level: 1


#### Столбец income_part

In [403]:
data["income_part"].unique()

array(['ИТОГО ДОХОДОВ ', 'НАЛОГОВЫЕ И НЕНАЛОГОВЫЕ ДОХОДЫ',
       'НАЛОГИ НА ПРИБЫЛЬ, ДОХОДЫ', 'Налог на прибыль организаций',
       'Налог на доходы физических лиц',
       'НАЛОГИ НА ТОВАРЫ (РАБОТЫ, УСЛУГИ), РЕАЛИЗУЕМЫЕ НА ТЕРРИТОРИИ РОССИЙСКОЙ ФЕДЕРАЦИИ',
       'Акцизы по подакцизным товарам (продукции), производимым на территории Российской Федерации',
       'НАЛОГИ НА СОВОКУПНЫЙ ДОХОД',
       'Налог, взимаемый в связи с применением упрощенной системы налогообложения',
       'Единый налог на вмененный доход для отдельных видов деятельности',
       'Единый сельскохозяйственный налог',
       'Налог, взимаемый в связи с применением патентной системы налогообложения',
       'НАЛОГИ НА ИМУЩЕСТВО', 'Налог на имущество физических лиц',
       'Налог на имущество организаций', 'Транспортный налог',
       'Налог на игорный бизнес', 'Земельный налог',
       'НАЛОГИ, СБОРЫ И РЕГУЛЯРНЫЕ ПЛАТЕЖИ ЗА ПОЛЬЗОВАНИЕ ПРИРОДНЫМИ РЕСУРСАМИ',
       'Налог на добычу полезных ископаемых',
      

In [404]:
parts = ["итого", "итого доходов", "итого доходов ", " итого доходов"]
for part in parts:
    print(f"Количество записей содеражих \"{part}\" в обработанном столбце income_part: {sum(data["income_part"].str.lower().str.strip().str.contains(part, na=False))}")

Количество записей содеражих "итого" в обработанном столбце income_part: 10706
Количество записей содеражих "итого доходов" в обработанном столбце income_part: 10706
Количество записей содеражих "итого доходов " в обработанном столбце income_part: 0
Количество записей содеражих " итого доходов" в обработанном столбце income_part: 0


## Очистка датасета

**Неинформативные данные:**
- income_level — удаеляем, т.к. этот признак принимает абстрактные значения [0, 1, 2, 3] и подробно не раскрывает итоговую сумму дохода
- income_part — каждый месяц, каждого года есть запись со значением \"ИТОГО ДОХОДОВ\", учитываем только ее, остальное удаляем
- okato — удаеляем, т.к. это просто общероссийский классификатор объектов административно-территориального делени (Код региона/района/города/села по административному делению)<br>
- oktmo — удаеляем, т.к. это просто общероссийский классификатор территорий муниципальных образований (Код территории по муниципальному делени)
- object_level — удаеляем, т.к. этот признак имеет единственное уникальное значение

In [430]:
df = data.copy()
df = df[df["income_part"].str.lower().str.contains("итого доходов", na=False)]
df = df.drop(columns=["income_level", "income_part","object_level", "okato", "oktmo"])
df

,year,month,plan,execution_regional,growth_rate_regional,object_name
0,2015,5,6.579067e+07,2.468728e+07,1.108331,Белгородская область
385,2015,5,4.759697e+07,1.831169e+07,1.177592,Брянская область
761,2015,5,5.451191e+07,1.880686e+07,1.075378,Владимирская область
1190,2015,5,8.766248e+07,3.327730e+07,1.104055,Воронежская область
1563,2015,5,3.792311e+07,1.256591e+07,0.842478,Ивановская область
...,...,...,...,...,...,...
5857837,2025,10,2.116285e+08,1.732633e+08,1.177725,Республика Дагестан
5858609,2025,10,3.756894e+07,3.015028e+07,1.031745,Республика Ингушетия
5859001,2025,10,5.507933e+07,4.193369e+07,1.011666,Республика Северная Осетия — Алания
5859590,2025,10,2.295916e+08,1.649169e+08,1.077115,Ставропольский край


In [431]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10706 entries, 0 to 5860354
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   year                  10706 non-null  int64  
 1   month                 10706 non-null  int64  
 2   plan                  10706 non-null  float64
 3   execution_regional    10706 non-null  float64
 4   growth_rate_regional  10702 non-null  float64
 5   object_name           10706 non-null  object 
dtypes: float64(3), int64(2), object(1)
memory usage: 585.5+ KB


In [432]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10702 entries, 0 to 5860354
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   year                  10702 non-null  int64  
 1   month                 10702 non-null  int64  
 2   plan                  10702 non-null  float64
 3   execution_regional    10702 non-null  float64
 4   growth_rate_regional  10702 non-null  float64
 5   object_name           10702 non-null  object 
dtypes: float64(3), int64(2), object(1)
memory usage: 585.3+ KB


## Дополнение датасета

In [433]:
dct0 = {"Ханты-Мансийский автономный округ — Югра": "Ханты-Мансийский (Югра) автономный округ",
       "Удмуртская Республика": "Республика Удмуртия",
       "Чувашская Республика": "Республика Чувашия",
       "Кабардино-Балкарская Республика": "Республика Кабардино-Балкария",
       "Карачаево-Черкесская Республика": "Республика Карачаево-Черкесия",
       "Республика Северная Осетия — Алания": "Республика Северная Осетия - Алания",
       "Чеченская Республика": "Республика Чечня"}

df["object_name"] = df["object_name"].replace(dct0)

C:\Users\Maxim\AppData\Local\Temp\ipykernel_19868\1115743285.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["object_name"] = df["object_name"].replace(dct0)


In [434]:
df["date"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)
df

C:\Users\Maxim\AppData\Local\Temp\ipykernel_19868\3504982393.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["date"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)


,year,month,plan,execution_regional,growth_rate_regional,object_name,date
0,2015,5,6.579067e+07,2.468728e+07,1.108331,Белгородская область,2015-05
385,2015,5,4.759697e+07,1.831169e+07,1.177592,Брянская область,2015-05
761,2015,5,5.451191e+07,1.880686e+07,1.075378,Владимирская область,2015-05
1190,2015,5,8.766248e+07,3.327730e+07,1.104055,Воронежская область,2015-05
1563,2015,5,3.792311e+07,1.256591e+07,0.842478,Ивановская область,2015-05
...,...,...,...,...,...,...,...
5857837,2025,10,2.116285e+08,1.732633e+08,1.177725,Республика Дагестан,2025-10
5858609,2025,10,3.756894e+07,3.015028e+07,1.031745,Республика Ингушетия,2025-10
5859001,2025,10,5.507933e+07,4.193369e+07,1.011666,Республика Северная Осетия - Алания,2025-10
5859590,2025,10,2.295916e+08,1.649169e+08,1.077115,Ставропольский край,2025-10


## Проверка датасета

In [435]:
arr_0 = df["object_name"].unique()
for el in arr_0:
    if el not in arr_1:
        print(el, "-->", el in arr_1)

In [436]:
arr_1 = df1["region"].unique()
for el in arr_1:
    if el not in arr_0:
        print(el, "-->", el in arr_0)

## Экспорт датасета

In [437]:
out_name = "regions_income_with_dates.csv"
df.to_csv(out_name, sep=";", index=False, encoding="utf-8-sig")
print(f"Файл сохранён: {out_name}")

Файл сохранён: regions_income_with_dates.csv
